Для начала проведем анализ наших данных

In [ ]:
import re

import pandas as pd

In [ ]:
df_train = pd.read_parquet('dataset/train.parquet')
df_items = pd.read_parquet('dataset/benchmark_items.parquet')
df_queries = pd.read_parquet('dataset/benchmark_queries.parquet')

In [ ]:
df_train.head()

In [ ]:
df_items.head()

In [ ]:
df_items.loc[4, 'item_description_raw']

In [ ]:
df_items.loc[2, 'item_description_raw']

In [ ]:
df_items.loc[4, 'item_infm_params_text']

In [ ]:
df_items.loc[0, 'item_title_raw']

In [ ]:
df_items.loc[4, 'item_title_raw']

In [ ]:
df_items.loc[2, 'item_title_raw']

In [ ]:
df_queries.head()

In [ ]:
a = [i for i in df_items.item_description_raw.astype(str) if bool(re.search(r'[a-zA-Z]', i))]
a

Если мы посмотрим на то, как именно представлены текстовые данные в таблицах, то можно предположить, что для данной задачи лучше всего будет использовать TF-IDF, точнее его более точную версию BM25.

1) Почему не ембеддинги?
По моему мнению ембеддинги здесь будут неуместны , потому что у нас сломанный контекс , сам запрос пользователя - это просто короткое сообщение, которое мало чего отражает.

2) Почему BM25, а не TF-IDF ?
BM25 улучшенная версия TF-IDF, если параметры регулирующие вклад повторяющихся слов и длины текста прировнять определенным значениям , то мы можем поулчить TF-IDF.

Еще важный момент:
надо предобработать текст , во-первых, привести к нижнему регистру , убрать стоп слова, лемматизировать, убрать знак препинания, убрать странные символы и эмодзи, еще проблема в английских словах , пользователь может их написать на кириллице, или наоборот.


---

In [ ]:
!pip install bm25s catboost pymorphy3 pymorphy3-dicts-ru nltk

In [ ]:

import re
import pandas as pd
import numpy as np
import pymorphy3
import nltk
from nltk.corpus import stopwords
from tqdm.auto import tqdm

morph = pymorphy3.MorphAnalyzer()
lemma_cache = {}

TEXT_COLS = ['search_query', 'search_infm_params_text', 'item_title_raw', 'item_description_raw', 'item_infm_params_text']
ID_COLS = ['item_id', 'query_id', 'search_location_id', 'search_category', 'item_category_id', 'item_microcat_id', 'item_location_id']
NUM_COLS = ['item_price', 'item_rating', 'item_rating_reviews_count', 'item_is_phone_hidden', 'item_is_message_forbidden',
            'item_latitude', 'item_longitude', 'search_is_delivery_search']


def clean_text(s):
    # нижний регистр, ё -> е, всё кроме букв и цифр (эмодзи, пунктуация, неизвестные символы) -> пробел
    s = s.lower().replace('ё', 'е')
    return re.sub(r'[^а-яa-z0-9]+', ' ', s).strip()


def lemma(w):
    if w not in lemma_cache:
        lemma_cache[w] = morph.parse(w)[0].normal_form.replace('ё', 'е') if re.search('[а-я]', w) else w
    return lemma_cache[w]


nltk.download('stopwords', quiet=True)
nltk_stop = [w.replace('ё', 'е') for w in stopwords.words('russian')]
STOPWORDS = set(nltk_stop) | {lemma(w) for w in nltk_stop}


def lemmatize(s, max_chars=None):
    # чистка -> леммы -> без стоп-слов и одиночных букв
    if max_chars:
        s = s[:max_chars]
    lemmas = [lemma(w) for w in clean_text(s).split()]
    return [l for l in lemmas if l not in STOPWORDS and (len(l) > 1 or l.isdigit())]


def lemmatize_col(s):
    m = {t: ' '.join(lemmatize(t)) for t in tqdm(s.unique(), leave=False)}
    return s.map(m)


def to_str_id(s):
    if pd.api.types.is_numeric_dtype(s):
        s = s.astype('Int64') 
    return s.astype(str).replace({'<NA>': 'nan', 'None': 'nan'})


def normalize(df):
    # пропуски в тексте -> '', id -> строки, числа -> float
    df = df.copy()
    for c in TEXT_COLS:
        if c in df:
            df[c] = df[c].fillna('').astype(str)
    for c in ID_COLS:
        if c in df:
            df[c] = to_str_id(df[c])
    for c in NUM_COLS:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df


print(lemmatize('Скупаем телевизоры 📺 и ноутбуки!!! Дорого, быстро ★'))

['скупать', 'телевизор', 'ноутбук', 'дорого', 'быстро']


In [ ]:
import pandas as pd
import numpy as np
import bm25s
from catboost import CatBoostRanker, Pool
import gc

# ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
print("1. Загрузка данных...")
train = normalize(pd.read_parquet('dataset/train.parquet'))
queries = normalize(pd.read_parquet('dataset/benchmark_queries.parquet'))
items = normalize(pd.read_parquet('dataset/benchmark_items.parquet'))


items['text'] = [' '.join(lemmatize(t + ' ' + p) + lemmatize(d, 4000))
                 for t, p, d in tqdm(zip(items['item_title_raw'], items['item_infm_params_text'],
                                         items['item_description_raw']), total=len(items))]

queries['text'] = lemmatize_col(queries['search_query'] + ' ' + queries['search_infm_params_text'])
train['text'] = lemmatize_col(train['search_query'] + ' ' + train['search_infm_params_text'])

# ==========================================
# ПОСТРОЕНИЕ ИНДЕКСА BM25s
print("2. Индексация корпуса BM25s...")
corpus_tokens = bm25s.tokenize(items['text'].tolist())

retriever = bm25s.BM25(k1=1.2, b=0.5) # основные гиперпараметры , над которыми игрался без страха , что не хватит мощнастей, но прироста какого-то особо не дают
retriever.index(corpus_tokens)

def get_bm25_candidates(query_texts, k=2000):
    """Вспомогательная функция для пакетного поиска кандидатов"""
    query_tokens = bm25s.tokenize(query_texts.tolist())
    doc_indices, scores = retriever.retrieve(query_tokens, k=k)
    item_ids_array = items['item_id'].values[doc_indices]
    return item_ids_array, scores

# ==========================================
# 3. ПОДГОТОВКА ДАТАСЕТА ДЛЯ ОБУЧЕНИЯ (TRAIN)
print("3. Генерация кандидатов для обучения...")
Q_COLS = ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
train['query_id'] = 'tr_' + train.groupby(Q_COLS, sort=False, dropna=False).ngroup().astype(str)
train_q = train.drop_duplicates('query_id')

# Берем сэмпл запросов, чтобы не переполнить память
train_sample = train_q.sample(min(50000, len(train_q)), random_state=42).reset_index(drop=True)

# Для трейна берем топ-K кандидатов BM25, если брал больше , то у меня просто все дропалось , мощнастей не хватало 
K_TRAIN = 1000
train_candidates, train_scores = get_bm25_candidates(train_sample['text'], k=K_TRAIN)

print("Сборка плоской таблицы (Flat Table) для трейна...")
# Каждая пара "запрос — кандидат" с НАСТОЯЩИМ скором BM25
df_train = pd.DataFrame({
    'query_id': np.repeat(train_sample['query_id'].values, K_TRAIN),
    'item_id': train_candidates.ravel(),
    'bm25_score': train_scores.ravel(),
})

# Метка: кандидат выбран пользователем по этому запросу
pos_pairs = train[['query_id', 'item_id']].drop_duplicates().assign(target=1)
df_train = df_train.merge(pos_pairs, on=['query_id', 'item_id'], how='left')
df_train['target'] = df_train['target'].fillna(0).astype(int)

df_train = df_train[df_train.groupby('query_id')['target'].transform('max') > 0]
print('запросов с позитивом в топ-%d: %d из %d' % (K_TRAIN, df_train['query_id'].nunique(), len(train_sample)))

# Очищаем память
del train_candidates, train_scores
gc.collect()

# ==========================================
# 4. ИНЖИНИРИНГ ПРИЗНАКОВ
print("4. Обогащение признаками...")
# Отбираем числовые и категориальные фичи из объявлений
item_features = items[['item_id', 'item_price', 'item_rating', 'item_rating_reviews_count',
                       'item_location_id', 'item_category_id', 'item_microcat_id']]

#  search_category и item_category_id значения из разных справочников, по бэнефиту будет не напрямую их сравнивать 
# (признак был бы всегда 0). Вместо этого по train будем считать, как часто при поиске в категории
# search_category выбирают объявления из item_category_id / item_microcat_id.
# Статистику берём по train, на которых учим ранкер, чтобы не было data leakage.
stats = train[~train['query_id'].isin(train_sample['query_id'])]


def category_prior(col):
    cnt = stats.groupby(['search_category', col]).size()
    return (cnt / cnt.groupby(level=0).transform('sum')).rename('p_' + col).reset_index()


cat_prior = category_prior('item_category_id')      
micro_prior = category_prior('item_microcat_id')    

top_cat = (cat_prior.sort_values('p_item_category_id')
           .drop_duplicates('search_category', keep='last')
           [['search_category', 'item_category_id']]
           .rename(columns={'item_category_id': 'top_category'}))


def add_features(df, query_features):
    df = df.merge(item_features, on='item_id', how='left')
    df = df.merge(query_features, on='query_id', how='left')
    df = df.merge(cat_prior, on=['search_category', 'item_category_id'], how='left')
    df = df.merge(micro_prior, on=['search_category', 'item_microcat_id'], how='left')
    df = df.merge(top_cat, on='search_category', how='left')

    # Кросс-фичи: совпадение локации и "объявление из самой частой категории для этого поиска"
    df['is_location_match'] = (df['item_location_id'] == df['search_location_id']).astype(int)
    df['is_category_match'] = (df['item_category_id'] == df['top_category']).astype(int)

    # Заполняем пропуски в числовых фичах
    for c in ['item_price', 'item_rating', 'item_rating_reviews_count', 'p_item_category_id', 'p_item_microcat_id']:
        df[c] = df[c].fillna(0)
    return df


df_train = add_features(df_train, train_sample[['query_id', 'search_location_id', 'search_category']])

# сортируем group_id для CarBoost
df_train = df_train.sort_values('query_id').reset_index(drop=True)
print('доля is_category_match=1 среди позитивов:', round(df_train.loc[df_train.target == 1, 'is_category_match'].mean(), 3))

# ==========================================
# 5. ОБУЧЕНИЕ CATBOOSTRANKER
print("5. Обучение модели ранжирования (YetiRank)...")
features = ['bm25_score', 'item_price', 'item_rating', 'item_rating_reviews_count',
            'is_location_match', 'is_category_match', 'p_item_category_id', 'p_item_microcat_id']

train_pool = Pool(
    data=df_train[features],
    label=df_train['target'],
    group_id=df_train['query_id']
)

#Вот здесь можно поиграться с глубиной, количеством деревьев и шагом обучения, 
#но к сожалению у меня не хватило мощнастей, чтобы провести эксперименты, даже на колабе
ranker = CatBoostRanker( 
    loss_function='YetiRank',
    eval_metric='NDCG',
    iterations=800,
    depth=6,
    learning_rate=0.1,
    task_type='GPU',
    random_seed=42,
    verbose=100
)

ranker.fit(train_pool)

del df_train, train_pool, train_sample
gc.collect()

# ==========================================
# 6. ИНФЕРЕНС НА БЕНЧМАРКЕ
print("6. Генерация 2000 кандидатов для тестовых запросов...")
test_candidates, test_scores = get_bm25_candidates(queries['text'], k=2000) # взял 2000, чтобы захватить больше для более высокой оценки

print("Сборка таблицы инференса и предсказание...")
test_data = []
for i, q_id in enumerate(queries['query_id']):
    for cand_item, score in zip(test_candidates[i], test_scores[i]):
        test_data.append({'query_id': q_id, 'item_id': cand_item, 'bm25_score': score})

df_test = pd.DataFrame(test_data)

# Те же признаки, что и на трейне
df_test = add_features(df_test, queries[['query_id', 'search_location_id', 'search_category']])

# Получаем новые скоры от Кэтбуста
df_test['cb_score'] = ranker.predict(df_test[features])

# ==========================================
# 7. ФОРМИРОВАНИЕ ОТВЕТА
print("7. Подготовка файла с ответами...")
# Сортируем: сначала по запросу, затем по оценке Кэтбуста (по убыванию)
df_test = df_test.sort_values(['query_id', 'cb_score'], ascending=[True, False])

top_50_preds = df_test.groupby('query_id').head(50)

answer = top_50_preds.groupby('query_id')['item_id'].apply(lambda x: ' '.join(x.astype(str))).reset_index()
answer.rename(columns={'item_id': 'answer'}, inplace=True)

answer.to_csv('answer.csv', index=False)
print("Готово! Результат сохранен в answer.csv.")

1. Загрузка данных...


  0%|          | 0/189212 [00:00<?, ?it/s]

  0%|          | 0/2452 [00:00<?, ?it/s]

  0%|          | 0/107957 [00:00<?, ?it/s]

2. Индексация корпуса BM25s...


...

Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 126ms	remaining: 1m 40s
100:	total: 8.65s	remaining: 59.9s
200:	total: 14.1s	remaining: 41.9s
300:	total: 19s	remaining: 31.4s
400:	total: 24s	remaining: 23.9s
500:	total: 29.1s	remaining: 17.4s
600:	total: 34.3s	remaining: 11.4s
700:	total: 39.6s	remaining: 5.6s
799:	total: 44.9s	remaining: 0us
6. Генерация 2000 кандидатов для тестовых запросов...


Split strings:   0%|          | 0/2452 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/2452 [00:00<?, ?it/s]

Сборка таблицы инференса и предсказание...
7. Подготовка файла с ответами...
Готово! Результат сохранен в answer.csv.


In [ ]:
df = pd.read_csv('answer.csv')
df.head()

,query_id,answer
0,00WuFMaXSFZBxSzT,980953f92323e8d6 2f82afc9946e6c84 f3e39b2798e2...
1,03ztb1gtRFC4K4vP,a7452b3fb8825c43 6a920b36f8ad8241 a5afb8923b2d...
2,052l5ySkk2Asxdni,29d267e0809c4c9e 3b414fcfcca8c33d 38e74abf2fb3...
3,06QIixa5pbkQaLnl,92565f1eb6dab23e 22237f337584cb41 42fa06bf89c4...
4,0A2x7ulC3KohJSRe,53f7f5708169712c 105861cd5ae0e0f5 abefa4141c2c...


Что по итогу сделано:
1) Данные и признаки 
- Их я предобработал так , как писал выше
- Для самого ранжирования (ML-этап) закинул числовые и категориальные фичи: цену, рейтинг и количество отзывов. Еще нагенерил бинарных кросс-фичей - флаги совпадения по городу (is_location_match) и категории (is_category_match). Точное совпадение региона и категории критически важно, и для кэтбуста это прям сильный сигнал.

2) Примененные алгоритмы и модели
- Сначала юзал библиотеку bm25s как быстрый фильтр. Взял 1000 кандидатов на инференсе, чтобы захватить больше для более высокой оценки (полноты). Вот здесь еще можно поиграться с параметрами k1 и b под короткие тексты. (в комментариях в коде писал)
- Затем весь этот пул сужал до 50 через CatBoostRanker. В качестве лосса взял YetiRank, так как он напрямую оптимизирует саму выдачу (NDCG) и выталкивает нужное объявление прямо в наш итоговый топ-50.
- По поводу CatBoostRanker: есть много альтернатив, но я выбрал эту , потому что ее сделали русские, и я уже работал с catboost, а NDCG для данной задачи просто идеально здесь подходит как функция , через которую можно оптимизировать модель, а для этого как раз идеально подходит YetiRank(задает градиент недифференцируемой функции NDCG)

3) Проверка качества до отправки
- Вообще у меня изначально был другой ноутбук, там я проводил тесты, изначально разбивая выборку на треин и тест, но потом я заметил, что рост локальной метрики жестко коррелирует с результатами на степике, и из-за нехватки времени начал сразу туда заливать попытки.

4) Анализ ошибок и их исправление
- Самая главная проблема — нехватка оперативки при сборке плоской таблицы для обучения. К сожалению у меня не хватило мощнастей, чтобы провести эксперименты с большим пулом, даже на колабе если брал больше, то у меня просто все дропалось, мощнастей не хватало.
- Еще изначально бм25 пропускал кучу объявлений из-за регистра и мусорных символов. Как только добавил агрессивную очистку текста, метрика бустанулась почти на 30%. Также я пытался использовать классический TF-IDF, но он оказался значительно слабее BM25, также я игрался с гиперпараметрами BM25, но результат не менялся сильно, максимум на сотые. Еще я пытался свои реализации написать BM25 и TF-IDF, но библиотека работала значительно лучше и быстрее. 
- Также я все таки попытался использовать ембеддинги слов, вначале ruBert , но это ничего не дало , потом я взял sbert от сбера , но он тоже ничего не дал, хотя был лучше ruBert, все таки из-за того, что их дообучали на русских текстах, у них все равно есть представление о русских словах, можно конечно как-нибудь формализовать запрос пользователя , но это уже надо подключать модели. Я в итоге отказался от ембеддингов в пользу простоты кода , мб можно как-нибудь прикрутить их умом , но у меня не получилось.

5) Вывод 
- Что я вынес для себя: грамотная чистка текста на старте и пара сильных логичных фичей (типа совпадения по гео и категории) часто дают куда больше профита, чем попытки бесконечно усложнять саму модель. Пайплайн получился стабильным, рабочим и его можно скейлить, если перенести на машину помощнее.
Даже с жесткими ограничениями по оперативке на колабе удалось собрать рабочий пайплайн и получить довольно хороший Recall@50.

